### **1.   Instalando o Groq**

Inicialmente precisamos executar o comando %pip install groq para instalar o groq no nosso notebook já que utilizamos a API fornecida pelo groq cloude, ao escrever o comando basta executar para que a instalação seja feita



In [35]:
%pip install groq

### **2.   Importando a API key**


Anteriormente como foi orientado no vídeo, tivemos que criar uma conta no Groq para ao utilizar o Groq Cloude criar uma API Key que seria posteriomente utilizada no código, o comando abaixo faz uma importação da biblioteca "os" e dá o comando para preparar o ambiente com a utilização da chave que criamos no Grok Cloud

In [36]:
import os
os.environ['GROQ_API_KEY']="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"

### **3.   Importação do modelo**


Uitilizamos ums inicialização pronta presente no Groq Cloude para fazer a importação do modelo que será utilizado pelo nosso agente, no caso é o llama 70b, modelo recomendado pelo autor do vídeo

In [37]:
import os

from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Fast language models are essential for various applications in natural language processing (NLP) due to their ability to process and understand human language quickly and efficiently. The importance of fast language models can be seen in several areas:

1. **Real-time Applications**: Fast language models enable real-time applications such as chatbots, virtual assistants, and speech recognition systems. These models can respond promptly to user input, making the interaction more natural and engaging.
2. **Improved User Experience**: Fast language models can process large amounts of text data quickly, allowing for faster and more accurate search results, sentiment analysis, and text classification. This leads to a better user experience, as users can quickly obtain the information they need.
3. **Efficient Data Processing**: Fast language models can handle large volumes of data, making them ideal for applications that require processing vast amounts of text data, such as text analytics, 

### 4.   **Modelagem do Agente**
Iniciamos o processo de modelagem do Agente, para isso criamos uma classe Agent que irá interagir com o modelo. Para isso o Agent é inciado, são criados métodos para processar as mensagagens, para executar as solicitações e retornar uma resposta

In [38]:
class Agent:
  def __init__(self, client, system):
    self.client = client
    self.system = system
    self.messages = []
    if self.system is not None:
      self.messages.append({"role": "system", "content": self.system})

  def __call__(self, message=""):
    if message:
      self.messages.append({"role": "user", "content": message})
    result = self.execute()
    self.messages.append({"role": "assistant", "content": result})
    return result

  def execute(self):
    completetion = client.chat.completions.create(
        messages= self.messages,
        model="llama-3.3-70b-versatile",
    )
    return completetion.choices[0].message.content


### 5.   **Prompot do sistema**
Aqui definimos o comportamento do agente, a descrição do loop de pensamento, ação, pause e observação


In [39]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this:

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944×10e25.

Now it's your turn:
""".strip()

### 6.   **Criando as ferramentas**
São criadas as funções que irão realizar os cálculos e o autor chama essas funções de ferramentas as quais o agente irá consultar para obter uma determinada resposta no processo de ação. Em agentes associados a base de dados ou que fazem pesquisa em tempo real na internet, essas bases de dados seriam a ferramenta as quais eles consultariam para obter as respostas das quais precisam, como estamos fazendo um modelo mais singelo e inicial para aprendizagem, a nossa ferramenta são funções que irão gerar os resultados dos quais esse agente precisa.

In [40]:
def calculate(operation):
    return eval(operation)

def get_planet_mass(planet) -> float:
    match planet.lower():
        case "earth":
            return 5.972e24
        case "jupiter":
            return 1.898e27
        case "mars":
            return 6.39e23
        case "mercury":
            return 3.285e23
        case "neptune":
            return 1.024e26
        case "saturn":
            return 5.683e26
        case "uranus":
            return 8.681e25
        case "venus":
            return 4.867e24
        case _:
            return 0.0

### 7.   **Criando o loop**

A função do loop é para que o Agente "pense" de forma autonoma sem precisar acionar o comando manualmente toda vez, ou seja, sem que o usuário precise toda vez dar um comando em cada PAUSE para que o agente continue seu ciclo até obter a resposta desejada.

In [43]:
import re

def agent_loop(max_iterations, system, query):
  agent = Agent(client, system_prompt)
  tools = ["calculate", "get_planet_mass"]
  next_prompt = query
  i = 0

  while i < max_iterations:
      i += 1
      result = agent(next_prompt)
      print(result)

      if "PAUSE" in result and "Action" in result:
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            arg = action[0][1]
            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = "Observation: Tool not found"

            print(next_prompt)
            continue

            if "Answer" in result:
              break

### 8.   **Chamando a função loop**

Por fim, após os passos anteriores e suas respectivas execuções, chamamos a função de loop que foi criada anteriormente para interagir com o agente e nesse caso com o obtivo de saber qual é a mass da terra somada a massa de mercurio, multiplicado por 5, definimos o maximo de iterações em 10 e com isso o agente faz de forma autonoma o processo por 10 iterações, mesmo observando que por algumas iterações esse resultado é apenas repetido e o agente não precisa de todas as iterações para obter a resposta final.


In [44]:
agent_loop(max_iterations=10, system=system_prompt, query="What is the mass of Earth plus the mass of Mercury and all of that times 5?")

Thought: To solve this problem, I need to find the masses of Earth and Mercury, add them together, and then multiply the result by 5. First, I'll get the mass of Earth.

Action: get_planet_mass: Earth
PAUSE
Observation: 5.972e+24
Thought: Now that I have the mass of Earth, I need to find the mass of Mercury and add it to the mass of Earth.

Action: get_planet_mass: Mercury
PAUSE
Observation: 3.285e+23
Thought: I now have the masses of both Earth and Mercury. I'll add these two masses together to get the total mass.

Action: calculate: 5.972e+24 + 3.285e+23
PAUSE
Observation: 6.300500000000001e+24
Thought: Now that I have the total mass of Earth and Mercury, I need to multiply this by 5 to get the final result.

Action: calculate: 6.300500000000001e+24 * 5
PAUSE
Observation: 3.1502500000000004e+25
Thought: I have now calculated the total mass of Earth and Mercury multiplied by 5, so I can output the final answer.

Answer: The mass of Earth plus the mass of Mercury and all of that times 